In [1]:
#FILE_PATH = 'C://Users//User//Downloads//bitmex_data_1m.csv'
FILE_PATH = 'D://bitmex_data_1m.csv'

In [46]:
from abc import ABC, abstractmethod
from common import *
import plotly

# 데이터 로드 전략 인터페이스
class DataLoaderStrategy(ABC):
    @abstractmethod
    def load_data(self):
        #1분봉 데이터를 불러오는 과정
        pass
    
    #@abstractmethod
    #def pre_precessing(self, df, base_delta, from_date=None):
    #    pass
    #    #1분봉 데이터를 여러 분봉으로 변경하는 함수
        

# 구체적인 데이터 로드 전략: CSV 로드
class BitmexCSVDataLoader(DataLoaderStrategy):
    
    raw_data = None #1분봉 df를 담을 변수
    final_df = None
    
    def __init__(self, base_delta: int, from_date:str):
        self.base_delta = base_delta  # 인스턴스 변수로 저장
        self.from_date = from_date
    
    def load_data(self):
        print("CSV 데이터를 로드하고 Nan을 제거합니다.")
        import pandas as pd
        df = pd.read_csv(FILE_PATH, delimiter=',')
        df = df.dropna()
        
        print("CSV 데이터 로드 완료.")
        
        df=df[['timestamp','high','low','open','close']]
        
        #timestamp 를 kst 로 조정
        df['timestamp_kst'] = df['timestamp'].apply(convert_gmt_to_kst)
        
        #클래스에 세팅
        raw_data = df
        print(f"입력받은 {self.base_delta}분봉으로 {self.from_date} 부터 표현합니다.")
        df = self.__pre_precessing(df, self.base_delta, self.from_date)
        print(f"전처리 완료")

        return df
    
    def __pre_precessing(self, df, base_delta , from_date=None):
        df = df[['timestamp_kst','open','low','high','close']]
    
        df = df.reset_index()
        
        #int 형태의 timestamp 열도 추가
        df['timestamp_int'] = df['timestamp_kst'].apply(convert_to_timestamp)
        
        #다시 필요한 컬럼만 정돈
        df = df[['timestamp_kst','timestamp_int','open', 'low','high','close']]
        
        #최종 x분봉의 형태구현 base_delta = 15, 45, 240, 1440
        final_df = df[['low']].rolling(window=base_delta).min()     
        final_df['open'] = df[['open']].rolling(window=base_delta).apply(lambda x: x[0], raw=True)
        final_df['high'] = df[['high']].rolling(window=base_delta).max() 
        final_df['close'] = df['close']                         
        final_df['timestamp_int'] = df[['timestamp_int']]-(base_delta*60-60)  
        
        final_df = final_df[base_delta-1:] #window수 -1 값만큼 버리고
        final_df['timestamp_kst'] = final_df['timestamp_int'].apply(convert_timestamp_to_datetime_str)
        final_df = final_df.reset_index()[['timestamp_kst','open','low','high','close','timestamp_int']] #리셋재구성
        
        final_df = final_df[final_df['timestamp_int']%(base_delta*60)==0]
        
        return final_df
        

# 데이터 처리 전략 인터페이스
class DataProcessingStrategy(ABC):
    @abstractmethod
    def process_data(self, data):
        pass

# 구체적인 데이터 처리 전략: 이동 평균 계산
class MovingAverageProcessing(DataProcessingStrategy):
    def process_data(self, data):
        print("이동 평균을 계산했습니다.")
        data['ma_20']=self.__sma(data,20)
        data['ma_60']=self.__sma(data,60)
        
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __sma(self, data, period=20):
        return data['close'].rolling(window=period, min_periods=1).mean()
    
# 구체적인 데이터 처리 전략: RSI 계산
class RSIProcessing(DataProcessingStrategy):
    def process_data(self, data):
        self.__rsi(data)
        print("RSI를 계산했습니다.")
        return data#[sum(data[:i+1])/(i+1) for i in range(len(data))]
    
    def __rsi(self, data, period=14):
    
        import numpy as np
        delta = data['close'].diff(1)  # 종가의 변화량 계산
        gain = np.where(delta > 0, delta, 0)  # 상승분
        loss = np.where(delta < 0, -delta, 0)  # 하락분
    
        avg_gain = pd.Series(gain).rolling(window=period, min_periods=1).mean()
        avg_loss = pd.Series(loss).rolling(window=period, min_periods=1).mean()
        
        rs = avg_gain / (avg_loss + 1e-10)  # 0으로 나누는 오류 방지
        rsi = 100 - (100 / (1 + rs))
        
        rsi.index = data.index
        
        return rsi

#고점을 찾는 처리 전략
class HighPointScoringProcessing(DataProcessingStrategy):
    '''
    메인 df에 'high_score' 라는 컬럼을 추가하고, 외부 변수의 리스트로 들어온 
    밴드 값을 shift 하면서 그중에 가장 큰 가격에 +1 스코어를 한다.
    '''
    
    
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
    
    def process_data(self, data):
        data = self.__get_high_score_by_list(data,self.bandwith_list)
        return data
    
    def __get_high_score_by_list(self, origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = self.__get_high_score_with_bandwidth(origin_df, 'high', i)
                else:
                    return_df = self.__get_high_score_with_bandwidth(return_df, 'high', i, reset=False)
        
        return return_df
    
    
        
    def __get_high_score_with_bandwidth(self, target_df, column_name, bandwidth, reset=True):
        '''
        df를 제공하면서 밴드 값을 같이 제공하면 이를 반복문으로 돌아가면서 score 를 쌓는 함수
        '''
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['high_score']=0
        end_index = len(target_df) - bandwidth
        for idx,i in enumerate(range(end_index+1)):
            
            max_index = target_df.iloc[i:i+bandwidth][column_name].idxmax()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
            
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
            
            #print(f"체크값 :{target_df.loc[max_row_index]['high_score']+1}")
            
            target_df.loc[max_index,'high_score'] = target_df.loc[max_index]['high_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
            
        
        
        result_df = target_df.copy(deep=True)
        
        print(f"수행한 숫자:{end_index}")
        print(f"가장 높은 점수:{result_df.iloc[result_df['high_score'].idxmax()]}")
        
        return result_df

#저점을 찾는 처리 전략
class LowPointScoringProcessing(DataProcessingStrategy):
    def __init__(self, bandwidth_list):
        self.bandwith_list = bandwidth_list
        
    def process_data(self, data):
        data = self.__get_low_score_by_list(data, self.bandwith_list)
        return data
    
    def __get_low_score_by_list(self,origin_df,lst):
        if(len(lst)==0):
            print("대상이 없습니다.")
            
        else:
            return_df = None
            for idx, i in enumerate(lst):
                if(idx==0):
                    return_df = get_low_score_with_bandwidth(origin_df, 'low', i)
                else:
                    return_df = get_low_score_with_bandwidth(return_df, 'low', i, reset=False)
        
        return return_df
    
    
    def __get_low_score_with_bandwidth(self,target_df, column_name, bandwidth, reset=True):
    
    
        #일단 들어온 df 에 high_score 라는 컬럼 값 기본 강제 삽입
        if(reset==True):
            target_df['low_score']=0
        end_index = len(target_df) - bandwidth
        for idx,i in enumerate(range(end_index+1)):
            
            min_index = target_df.iloc[i:i+bandwidth][column_name].idxmin()
            
            #print(f"진입한 시작 값 : {i}~{i+bandwidth}")
            #print(f"가장 큰 인덱스 : {max_index}")
            #print(f"이때의 이미 기록된 값:{target_df.loc[max_index, 'high_score']} ")
            #print(f"이때의 이미 기록될 값:{target_df.loc[max_index]['high_score']+1} ")
            #print(max_index)
            
            #print(f"세팅전 값 : {target_df.loc[max_index]['high_score']}")
            
            #print(f"체크값 :{target_df.loc[max_row_index]['high_score']+1}")
            
            target_df.loc[min_index,'low_score'] = target_df.loc[min_index]['low_score']+1
            #print(f"세팅후 값 : {target_df.loc[max_index]['high_score']}")
            
        
        
        result_df = target_df.copy(deep=True)
        return result_df
    
    

# 데이터 시각화 전략 인터페이스
class VisualizationStrategy(ABC):
    @abstractmethod
    def visualize(self, data):
        pass

# 구체적인 시각화 전략: 라인 그래프
class LineGraphVisualization(VisualizationStrategy):
    def visualize(self, data):
        print("데이터를 그래프로 시각화합니다.")
        pass

# 알림 전략 인터페이스
class NotificationStrategy(ABC):
    @abstractmethod
    def send_notification(self, message: str):
        pass

# 구체적인 알림 전략: 문자 메시지 전송
class SMSNotification(NotificationStrategy):
    def send_notification(self, message: str):
        print(f"문자 메시지 전송: {message}")

        
class PatternDetector(ABC):
    
    @abstractmethod
    def load(self,base_delta, from_date):
        pass
    
    @abstractmethod
    def add_sub_indicator(self,indicator_instance_list):
        pass
    
    @abstractmethod
    def execute(self):
        pass
        
        
# Bitmex 클래스 (컨텍스트)
class Bitmex(PatternDetector):
    #def __init__(self, data_loader: DataLoaderStrategy, processor: DataProcessingStrategy,
    #             visualizer: VisualizationStrategy, notifier: NotificationStrategy):
    #    self.data_loader = data_loader
    #    self.processor = processor
    #    self.visualizer = visualizer
    #    self.notifier = notifier
    #    self.data = None
        
    def __init__(self):
        pass
        
        
    def set_loader(self,data_loader : DataLoaderStrategy):
        self.data_loader = data_loader
    
    def load(self):
        self.data = self.data_loader.load_data()
        #self.data = self.data_loader.pre_precessing(self.data, 15, '2023-12-31 15:01:00')
        
    def set_processor(self, processor: DataProcessingStrategy):
        self.processor = processor
        
    def add_sub_indicator(self,indicator_instance_list):
        '''외부에서 주입받은 DataProcessingStrategy 중, 보조지표 추가하는 작업으로 정의된 클래스를 수행시킨다.'''
        for one_indicator in indicator_instance_list:
            #print("데이터확인")
            #print(self.data)
            self.data = one_indicator.process_data(self.data)
            
        
    
    def execute(self):
        processed_data = self.processor.process_data(self.data)
        self.visualizer.visualize(processed_data)
        if processed_data[-1] > 3:  # 특정 패턴 감지 예시
            self.notifier.send_notification("패턴이 감지되었습니다!")

# 사용 예시
#if __name__ == "__main__":
#    bitmex = Bitmex(CSVDataLoader(), MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#    bitmex.load()


In [3]:
#bitmex = Bitmex(, MovingAverageProcessing(), LineGraphVisualization(), SMSNotification())
#bitmex.load()

CSV 데이터를 로드하고 Nan을 제거합니다.
CSV 데이터 로드 완료.
입력받은 15분봉으로 2023-12-31 15:01:00 부터 표현합니다.
전처리 완료


In [47]:
bitmex = Bitmex()
bitmex.set_loader(BitmexCSVDataLoader(15,'2023-12-31 15:01:00'))
bitmex.load()

CSV 데이터를 로드하고 Nan을 제거합니다.
CSV 데이터 로드 완료.
입력받은 15분봉으로 2023-12-31 15:01:00 부터 표현합니다.
전처리 완료


In [48]:
indicator_list = [MovingAverageProcessing(), RSIProcessing()]
bitmex.add_sub_indicator(indicator_list)

데이터확인
               timestamp_kst      open       low      high     close  \
21       2015-09-26 01:45:00    236.13    235.44    236.10    235.75   
39       2015-09-26 03:45:00    236.21    235.60    236.21    236.10   
52       2015-09-26 05:45:00    235.97    235.03    236.15    235.03   
56       2015-09-26 06:00:00    236.00    234.92    235.66    235.22   
65       2015-09-26 06:45:00    235.08    234.81    235.22    234.90   
...                      ...       ...       ...       ...       ...   
3451726  2023-05-08 11:00:00  28308.00  28207.00  28374.00  28362.00   
3451741  2023-05-08 11:15:00  28362.00  28293.00  28374.00  28329.50   
3451756  2023-05-08 11:30:00  28329.50  28260.00  28330.50  28330.50   
3451771  2023-05-08 11:45:00  28330.50  28315.50  28395.00  28376.00   
3451786  2023-05-08 12:00:00  28376.00  28310.50  28377.50  28320.00   

         timestamp_int  
21        1.443200e+09  
39        1.443207e+09  
52        1.443214e+09  
56        1.443215e+09  
65  

In [50]:
indicator_list_more = [HighPointScoringProcessing([200])]
bitmex.add_sub_indicator(indicator_list_more)

NameError: name 'HighPointScoringProcessing' is not defined

In [49]:
bitmex.data

,timestamp_kst,open,low,high,close,timestamp_int,ma_20,ma_60
21,2015-09-26 01:45:00,236.13,235.44,236.10,235.75,1.443200e+09,235.750000,235.750000
39,2015-09-26 03:45:00,236.21,235.60,236.21,236.10,1.443207e+09,235.925000,235.925000
52,2015-09-26 05:45:00,235.97,235.03,236.15,235.03,1.443214e+09,235.626667,235.626667
56,2015-09-26 06:00:00,236.00,234.92,235.66,235.22,1.443215e+09,235.525000,235.525000
65,2015-09-26 06:45:00,235.08,234.81,235.22,234.90,1.443218e+09,235.400000,235.400000
...,...,...,...,...,...,...,...,...
3451726,2023-05-08 11:00:00,28308.00,28207.00,28374.00,28362.00,1.683511e+09,28675.150000,28868.091667
3451741,2023-05-08 11:15:00,28362.00,28293.00,28374.00,28329.50,1.683512e+09,28642.475000,28858.633333
3451756,2023-05-08 11:30:00,28329.50,28260.00,28330.50,28330.50,1.683513e+09,28614.275000,28850.125000
3451771,2023-05-08 11:45:00,28330.50,28315.50,28395.00,28376.00,1.683514e+09,28590.850000,28842.741667
